# Data Preprocessing
This notebook covers the data loading, exploration, and preprocessing steps for the NSL-KDD dataset.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. Define Column Names
columns = (['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 
            'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 
            'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 
            'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 
            'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 
            'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 
            'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 
            'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 
            'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 
            'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'attack', 'level'])


In [2]:
# 2. Load the Dataset
print("Loading datasets...")
train_df = pd.read_csv('data/KDDTrain+.txt', names=columns)
test_df = pd.read_csv('data/KDDTest+.txt', names=columns)

print(f"Training data shape: {train_df.shape}")
print(f"Testing data shape: {test_df.shape}")

# We don't need the 'level' column (it's a difficulty score in NSL-KDD, not a feature)
train_df.drop('level', axis=1, inplace=True)
test_df.drop('level', axis=1, inplace=True)

train_df.head()


Loading datasets...
Training data shape: (125973, 43)
Testing data shape: (22544, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,25,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal
1,0,udp,other,SF,146,0,0,0,0,0,...,1,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal
2,0,tcp,private,S0,0,0,0,0,0,0,...,26,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune
3,0,tcp,http,SF,232,8153,0,0,0,0,...,255,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal
4,0,tcp,http,SF,199,420,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal


In [3]:
# 3. Map Attacks to 5 Main Categories (Normal + 4 Attack Types)
attack_mapping = {
    'normal': 'normal',
    'apache2': 'dos', 'back': 'dos', 'land': 'dos', 'neptune': 'dos', 
    'mailbomb': 'dos', 'pod': 'dos', 'processtable': 'dos', 'smurf': 'dos', 
    'teardrop': 'dos', 'udpstorm': 'dos', 'worm': 'dos',
    'ipsweep': 'probe', 'mscan': 'probe', 'nmap': 'probe', 'portsweep': 'probe', 
    'saint': 'probe', 'satan': 'probe',
    'ftp_write': 'r2l', 'guess_passwd': 'r2l', 'httptunnel': 'r2l', 'imap': 'r2l', 
    'multihop': 'r2l', 'named': 'r2l', 'phf': 'r2l', 'sendmail': 'r2l', 
    'snmpgetattack': 'r2l', 'snmpguess': 'r2l', 'spy': 'r2l', 'warezclient': 'r2l', 
    'warezmaster': 'r2l', 'xlock': 'r2l', 'xsnoop': 'r2l',
    'buffer_overflow': 'u2r', 'loadmodule': 'u2r', 'perl': 'u2r', 'ps': 'u2r', 
    'rootkit': 'u2r', 'sqlattack': 'u2r', 'xterm': 'u2r'
}

train_df['attack_class'] = train_df['attack'].map(lambda x: attack_mapping.get(x, 'unknown'))
test_df['attack_class'] = test_df['attack'].map(lambda x: attack_mapping.get(x, 'unknown'))

# Drop the original 'attack' column
train_df.drop('attack', axis=1, inplace=True)
test_df.drop('attack', axis=1, inplace=True)

print("Training set class distribution:")
print(train_df['attack_class'].value_counts())


Training set class distribution:
attack_class
normal    67343
dos       45927
probe     11656
r2l         995
u2r          52
Name: count, dtype: int64


In [4]:
# 4. Separate Features (X) and Labels (y)
X_train = train_df.drop('attack_class', axis=1)
y_train = train_df['attack_class']

X_test = test_df.drop('attack_class', axis=1)
y_test = test_df['attack_class']


In [5]:
# 5. One-Hot Encoding for Categorical Features
categorical_cols = ['protocol_type', 'service', 'flag']

# Combine train and test for consistent one-hot encoding columns
combined_X = pd.concat([X_train, X_test], axis=0)
combined_X = pd.get_dummies(combined_X, columns=categorical_cols, drop_first=False)

# Split them back
X_train = combined_X.iloc[:len(X_train), :]
X_test = combined_X.iloc[len(X_train):, :]

print(f"Features after one-hot encoding: {X_train.shape[1]}")


Features after one-hot encoding: 122


In [6]:
# 6. Label Encoding for the Target Variable
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)
print(f"Classes encoded as: {dict(zip(le.classes_, le.transform(le.classes_)))}")


Classes encoded as: {'dos': np.int64(0), 'normal': np.int64(1), 'probe': np.int64(2), 'r2l': np.int64(3), 'u2r': np.int64(4)}


In [7]:
# 7. Feature Scaling (Standardization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save preprocessed data
np.save('data/X_train.npy', X_train_scaled)
np.save('data/y_train.npy', y_train_encoded)
np.save('data/X_test.npy', X_test_scaled)
np.save('data/y_test.npy', y_test_encoded)
print("Preprocessing complete! Data saved to 'data/' folder.")


Preprocessing complete! Data saved to 'data/' folder.
